# NYC Taxi Fare Prediction - Phase 4: Feature Engineering
### Automatilda x New York City Taxi & Limousine Commision
---
**Goal:** Transform the cleaned dataset into a model-ready feature matrix.
Raw columns are replaced by engineered features that better capture the 
business signals identified in Phase 2 EDA.

**Operations (in order):**
1. Extract temporal features from pickup datetime
2. Compute trip duration in minutes
3. Apply log1p transforms to skewed continuous features
4. Bin passenger count into meaningful groups
5. One-hot encode categorical columns, binary flag payment type
6. Drop unused raw columns
7. Validate and save to ´data/taxi_features.parquet´

**Input:** `data/taxi_clean.parquet` - 991,998 rows x 12 columns
**Output:** `data/taxi_features.parquet` - 991,998 rows x ~18 columns

In [ ]:
# ------------------------------------------------------------
# Step 4.0: Notebook Initialization
# ------------------------------------------------------------

import sys
sys.path.append("..")

from src.config import *

# Load clean parquet
df = pd.read_parquet(CLEAN_DATA_PATH)

# Register with DuckDB
register_duckdb_table(df, table_name="trips_clean")

# Confirm baseline
print("\n" + "=" * 70)
print("Clean Dataset Baseline")
print("=" * 70)

print(f"   Rows    : {df.shape[0]:,}")
print(f"   Columns : {df.shape[1]}")
print(f"\n   Columns : {df.columns.tolist()}")

print("\n" + "=" * 70)
print("Dtypes")
print("=" * 70)

print(df.dtypes.to_string())


## Step 4.0: Notebook Initialization ✅

Clean dataset loaded successfully from `data/taxi_clean.parquet`.

| Metric | Value |
|---|---|
| Rows | 991,998 |
| Columns | 12 |
| Missing values | 0 |
| Datetime cols parsed | ✅ `datetime64[ns]` |

## Step 4.1: Temporal Feature Extraction
Extract time-based signals from `tpep_pickup_datetime`. These features capture the hourly, daily and rush-hour demand patterns identified in Phase 2, signals that trip distance alone cannot encode.

In [ ]:
# ------------------------------------------------------------
# Step 4.1: Notebook Initialization
# ------------------------------------------------------------

# Rush hour windows confirmed from Phase 2 Step 2.4 analysis
RUSH_HOUR_MORNING : tuple[int, int] = (7,  9)   # 07:00–09:59
RUSH_HOUR_EVENING : tuple[int, int] = (17, 20)  # 17:00–20:59
OVERNIGHT_WINDOW  : tuple[int, int] = (0,  5)   # 00:00–05:59

def extract_temporal_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Extract temporal features from the pickup datetime column.

    New columns created:
        - ``pickup_hour``: hour of day (0-23)
        - ``pickup_day_of_week``: day of week (0=Monday, ..., 6=Sunday)
        - ``is_weekend``: 1 if Saturday or Sunday, else 0
        - ``is_rush_hour``: 1 if morning (7-9) or evening (17-20) rush, else 0
        - ``is_overnight``: 1 if between midnight and 05:59, else 0

    The raw datetime columns are retained at this stage and dropped
    later in Step 4, after all features that depend on them are created.

    Parameters
    ----------
    df: pd.DataFrame
        DataFrame containing ``tpep_pickup_datetime`` as datetime64.

    Returns
    -------
    pd.DataFrame
        Copy of the DataFrame with five temporal feature columns appended.
    """
    df = df.copy()

    pickup = df["tpep_pickup_datetime"]

    df["pickup_hour"] = pickup.dt.hour
    df["pickup_day_of_week"] = pickup.dt.dayofweek    # 0=Monday, ..., 6=Sunday

    df["is_weekend"] = pickup.dt.dayofweek.isin([5, 6]).astype(int)

    df["is_rush_hour"] = (
        pickup.dt.hour.between(RUSH_HOUR_MORNING[0], RUSH_HOUR_MORNING[1])
        | pickup.dt.hour.between(RUSH_HOUR_EVENING[0], RUSH_HOUR_EVENING[1])
    ).astype(int)

    df["is_overnight"] = (
        pickup.dt.hour.between(OVERNIGHT_WINDOW[0], OVERNIGHT_WINDOW[1])
    ).astype(int)

    new_cols = [
        "pickup_hour", "pickup_day_of_week",
        "is_weekend", "is_rush_hour", "is_overnight"
    ]

    logger.info(f"Temporal features added: {new_cols}")

    return df

# Run
df = extract_temporal_features(df)

# Verify
temporal_cols = [
    "pickup_hour", "pickup_day_of_week",
    "is_weekend", "is_rush_hour", "is_overnight"
]

print("\n" + "=" * 70)
print("Temporal Feature Summary")
print("=" * 70)
print(f"\n  {'Feature':<22}  {'dtype':<12}  {'min':>5}  {'max':>5}  "
     f"{'mean':>7}  {'value_counts (top 3)'}")
print("-" * 95)

for col in temporal_cols:
    s = df[col]
    vc_str = " | ".join(
        f"{k}:{v:,}" for k, v in s.value_counts().head(3).items()
    )
    print(f"  {col:<22}  {str(s.dtype):<12}  {s.min():>5} {s.max():>5}  "
         f"{s.mean():>7.3f}  {vc_str}")

print("\n" + "=" * 70)
print("Binary Flag Rates")
print("=" * 70)
for col in ["is_weekend", "is_rush_hour", "is_overnight"]:
    rate = df[col].mean() * 100
    print(f"  {col:<18} {rate:.1f}% of trips")

print("\n" + "=" * 70)
print("SQL Cross-Check")
print("=" * 70)
register_duckdb_table(df, table_name="trips_feat")
print(quick_sql("""
    SELECT
        is_rush_hour,
        is_overnight,
        is_weekend,
        COUNT(*) AS trip_count,
        ROUND(AVG(fare_amount), 2) AS avg_fare
    FROM trips_feat
    GROUP BY 1, 2, 3
    ORDER BY avg_fare DESC
""").to_string(index=False))


## Step 4.1: Temporal Feature Extraction ✅

Five temporal features extracted from `tpep_pickup_datetime`.

### Feature Summary

| Feature | Dtype | Min | Max | Mean | Notes |
|---|---|---|---|---|---|
| `pickup_hour` | `int32` | 0 | 23 | 13.45 | Peak at 18:00 (60,236 trips) |
| `pickup_day_of_week` | `int32` | 0 | 6 | 3.60 | Saturday most frequent |
| `is_weekend` | `int64` | 0 | 1 | 0.482 | 48.2\% of trips |
| `is_rush_hour` | `int64` | 0 | 1 | 0.351 | 35.1\% of trips |
| `is_overnight` | `int64` | 0 | 1 | 0.130 | 13.0\% of trips |

### Binary Flag Rates

| Flag | % of Trips |
|---|---|
| `is_weekend` | 48.2\% |
| `is_rush_hour` | 35.1\% |
| `is_overnight` | 13.0\% |

### Average Fare by Temporal Segment

| Rush Hour | Overnight | Weekend | Trips | Avg Fare |
|---|---|---|---|---|
| 0 | 1 | 0 | 30,751 | \\$14.24 |
| 0 | 0 | 1 | 240,100 | \\$13.40 |
| 0 | 0 | 0 | 275,458 | \\$13.35 |
| 1 | 0 | 1 | 140,632 | \\$12.96 |
| 0 | 1 | 1 | 97,840 | \\$12.80 |
| 1 | 0 | 0 | 207,217 | \\$12.46 |

**Observations:**
- The Phase 2 finding is confirmed: **overnight weekday trips command the highest average fare (\\$14.24)**, which is consistent with longer airport and outer-borough runs in the pre-dawn hours
- Rush hour trips show the **lowest average fare (\\$12.46 on weekdays)**, reflecting the dominance of short intra-Manhattan commutes during peak hours
- Weekend non-rush trips sit in the middle (\\$13.40), a mix of leisure and longer daytime trips
- `is_rush_hour` and `is_overnight` are mutually exclusive by definition (no overlap in the hour windows), which is confirmed by the SQL output

## Step 4.2: Trip Duration Feature
Compute trip duration in minutes from the pickup and dropoff datetimes. Duration is an independent fare signal, a slow crosstown trip may cover little distance but still accumulate time-based meter charges.

In [ ]:
# ------------------------------------------------------------
# Step 4.2: Trip Duration Feature
# ------------------------------------------------------------

DURATION_MIN_MINUTES: float = 0.5    # trips under 30 seconds are likely errors
DURATION_MAX_MINUTES: float = 180.0  # trips over 3 hours are extreme outliers

def compute_trip_duration(
    df: pd.DataFrame,
    clip_limits: tuple[float, float] = (DURATION_MIN_MINUTES, DURATION_MAX_MINUTES)
) -> pd.DataFrame:
    """
    Compute trip duration in minutes from pickup and dropoff datetimes.

    Duration is clipped to the range [clip_limits[0], clip_limits[1]] to
    remove sub-minute recording errors and implausibly long trips. Clipped
    values are logged so the volume of affected rows is transparent.

    Parameters
    ----------
    df: pd.DataFrame
        DataFrame containing ``tpep_pickup_datetime`` and
        ``tpep_dropoff_datetime`` as datetime64 columns.
    clip_limits: tuple[float, float], optional
        (min_minutes, max_minutes) bounds for clipping duration.
        Defaults to (0.5, 180.0).

    Returns
    -------
    pd.DataFrame
        Copy of the DataFrame with ``trip_duration_min`` appended.
    """
    df = df.copy()

    duration = (
        df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]
    ).dt.total_seconds() / 60.0

    # Count anomalies before clipping
    n_below = int((duration < clip_limits[0]).sum())
    n_above = int((duration > clip_limits[1]).sum())

    df["trip_duration_min"] = duration.clip(
        lower=clip_limits[0],
        upper=clip_limits[1]
    )

    logger.info("trip_duration_min created")
    logger.info(f"  Clipped below {clip_limits[0]} min: {n_below:,} rows")
    logger.info(f"  Clipped above {clip_limits[1]} min: {n_above:,} rows")

    return df

# Run
df = compute_trip_duration(df)

# Verify
dur = df["trip_duration_min"]

print("\n" + "=" * 70)
print("Trip Duration Statistics")
print("=" * 70)
print(f"  Min: {dur.min():.2f} min")
print(f"  p25: {dur.quantile(0.25):.2f} min")
print(f"  Median: {dur.median():.2f} min")
print(f"  Mean: {dur.mean():.2f} min")
print(f"  p75: {dur.quantile(0.75):.2f} min")
print(f"  p95: {dur.quantile(0.95):.2f} min")
print(f"  Max: {dur.max():.2f} min")
print(f"  Std: {dur.std():.2f} min")

print("\n" + "=" * 70)
print("Correlation with fare_amount")
print("=" * 70)
r = df[["trip_duration_min", "fare_amount"]].corr().iloc[0, 1]
print(f"  Pearson r (duration vs fare): {r:.4f}")

print("\n" + "=" * 70)
print("Duration Bins (rough distribution")
print("=" * 70)
bins = [0, 5, 10, 15, 20, 30, 60, 180]
labels = ["0–5", "5–10", "10–15", "15–20", "20–30", "30–60", "60–180"]
binned = pd.cut(dur, bins=bins, labels=labels)
vc = binned.value_counts().sort_index()

for label, count in vc.items():
    pct = count / len(df) * 100
    print(f"  {label:<10} min : {count:>8,}  ({pct:.1f}%)")


## Step 4.2: Trip Duration Feature ✅

`trip_duration_min` computed from the difference between dropoff and pickup datetimes, clipped to [0.5, 180.0] minutes.

### Clipping Summary

| Clip Boundary | Rows Affected | Interpretation |
|---|---|---|
| Below 0.5 min | 1,585 | Sub-minute trips — meter errors or immediate cancellations |
| Above 180.0 min | 1,991 | Implausibly long trips — likely meter left running |

### Duration Statistics

| Metric | Value |
|---|---|
| Min | 0.50 min |
| p25 | 6.48 min |
| Median | 10.83 min |
| Mean | 14.09 min |
| p75 | 17.72 min |
| p95 | 35.28 min |
| Max | 180.00 min |
| Std | 13.08 min |

### Correlation with `fare_amount`

| Feature | Pearson r | Strength |
|---|---|---|
| `trip_duration_min` | 0.7186 | Strong positive |

### Duration Distribution

| Bin (minutes) | Trips | % |
|---|---|---|
| 0–5 | 157,344 | 15.9% |
| 5–10 | 296,650 | 29.9% |
| 10–15 | 212,552 | 21.4% |
| 15–20 | 127,791 | 12.9% |
| 20–30 | 120,072 | 12.1% |
| 30–60 | 70,610 | 7.1% |
| 60–180 | 6,979 | 0.7% |

**Observations:**
- At **r = 0.7186**, `trip_duration_min` is a strong fare predictor, second only to `trip_distance` (r=0.900) among legitimate features
- The bulk of trips (67.2%) fall in the 0–15 minute window, consistent with the short intra-Manhattan trip profile seen throughout EDA
- The mean (14.09 min) is notably higher than the median (10.83 min), confirming right skew driven by the long-duration tail
- 3,576 rows were clipped in total (0.36% of the dataset), a negligible volume that does not distort the distribution

## Step 4.3: Log1p Transforms
Apply log1p transformation to right-skewed continuous features. This compresses the long right tail, stabilizes variance, and brings the
distributions closer to normal, improving linear model performance and reducing the influence of extreme values on tree-based models.

Features transformed: `trip_distance`, `tolls_amount`, `trip_duration_min`

In [ ]:
# ------------------------------------------------------------
# Step 4.3: Log1p Transforms
# ------------------------------------------------------------

LOG1P_COLS: list[str] = ["trip_distance", "tolls_amount", "trip_duration_min"]

def apply_log1p_transforms(
    df: pd.DataFrame,
    cols: list[str]
) -> pd.DataFrame:
    """
    Apply log1p transformation to specified numeric columns.

    For each oclumn in ``cols``, a new column named ``log1p_{col}`` is created.
    The original column is retained at this stage and dropped later alongside
    the other raw columns.

    log1p(x) = log(1 + x) is used rather than log(x) to safely handle zero values
    (e.g. ``tolls_amount`` is zero for the majority of trips).

    Parameters
    ----------
    df: pd.DataFrame
        DataFrame containing the columns to transform.
    cols: list[str]
        Column names to apply log1p to.

    Returns
    -------
    pd.DataFrame
        Copy of the DataFrame with new ``log1p_{col}`` columns appended.

    Raises
    ------
    ValueError
        If any value in a target column is negative before transformation,
        as log1p is undefined for values below -1.
    """
    df = df.copy()

    for col in cols:
        if col not in df.columns:
            logger.warning(f"Column '{col}' not found, skipping.")
            continue

        n_negative = int((df[col] < 0).sum())
        if n_negative > 0:
            raise ValueError(
                f"Column '{col}' contains {n_negative:,} negative values. "
                f"log1p requires all values >= 0."
            )

        new_col = f"log1p_{col}"
        df[new_col] = np.log1p(df[col])

        skew_before = df[col].skew()
        skew_after = df[new_col].skew()

        logger.info(
            f"log1p('{col}') -> '{new_col}' | "
            f"skew: {skew_before:.3f} -> {skew_after:.3f}"
        )

    return df

# Run
df = apply_log1p_transforms(df, LOG1P_COLS)

# Verify
print("\n"+ "=" * 75)
print("Log1p Transform Summary")
print("=" * 75)

print(f"\n  {'Column':<22} {'Skew Before':>12} {'Skew After':>12} "
     f"{'Min After':>10} {'Max After':>10}")
print("-" * 70)

for col in LOG1P_COLS:
    new_col = f"log1p_{col}"
    print(
        f"  {col:<22} "
        f"{df[col].skew():>12.4f} "
        f"{df[new_col].skew():>12.4f} "
        f"{df[new_col].min():>10.4f} "
        f"{df[new_col].max():>10.4f}"
    )

print("\n" + "=" * 75)
print("Correlation with fare_amount (before vs after)")
print("=" * 75)

print(f"\n  {'Feature':<25} {'r (raw)':>10} {'r (log1p)':>10}")
print("-" * 50)

for col in LOG1P_COLS:
    new_col = f"log1p_{col}"
    r_raw = df[[col, "fare_amount"]].corr().iloc[0, 1]
    r_log1p = df[[new_col, "fare_amount"]].corr().iloc[0, 1]
    delta = r_log1p - r_raw

    print(f"  {col:<25} {r_raw:>10.4f} {r_log1p:>10.4f} (Δ {delta:+.4f})")


## Step 4.3: Log1p Transforms ✅

Log1p transformation applied to three right-skewed continuous features.

### Skewness Reduction

| Column | Skew Before | Skew After | Reduction | Min After | Max After |
|---|---|---|---|---|---|
| `trip_distance` | 3.194 | 1.111 | −65.2\% | 0.0100 | 5.1248 |
| `tolls_amount` | 10.996 | 3.914 | −64.4\% | 0.0000 | 5.4082 |
| `trip_duration_min` | 5.280 | 0.108 | −98.0\% | 0.4055 | 5.1985 |

### Correlation with `fare_amount` — Raw vs Log1p

| Feature | r (raw) | r (log1p) | Δ |
|---|---|---|---|
| `trip_distance` | 0.9441 | 0.8873 | −0.0568 |
| `tolls_amount` | 0.6058 | 0.6207 | +0.0149 |
| `trip_duration_min` | 0.7186 | 0.7428 | +0.0242 |

**Observations:**
- `log1p_trip_duration_min` achieves the most dramatic skew reduction (5.28 → 0.108, a 98% improvement), the distribution is now
  near-symmetric and well-suited for linear models
- `log1p_trip_distance` skew drops from 3.19 to 1.11, still mildly right-skewed due to the JFK flat-rate cluster, but substantially improved
- `log1p_tolls_amount` retains higher skew (3.91) due to the extreme zero-inflation, the majority of trips have zero tolls, which log1p
  maps to exactly 0.0, preserving the spike
- The slight **decrease** in correlation for `log1p_trip_distance` (−0.057) is expected: the raw linear relationship was already very
  strong (r=0.944), and log1p compresses the scale, moderately reducing the Pearson r while improving distributional properties
  for model fitting
- `log1p_trip_duration_min` and `log1p_tolls_amount` both show **improved** correlation after transformation, confirming the
  transform is beneficial for these features

## Step 4.4: Passenger Count Binning
Bin `passenger_count` into three meaningful groups identified in Phase 2.
The near-zero correlation with fare (r=0.016) means the raw count carries little signal — grouping into broad categories reduces noise while preserving any weak group-size effect.

In [ ]:
# ------------------------------------------------------------
# Step 4.4: Passenger Count Binning
# ------------------------------------------------------------

# Bin boundaries and labels confirmed from Phase 2 Step 2.3 analysis.
PASSENGER_BINS: list[int] = [0, 1, 3, 9]
PASSENGER_LABELS: list[str] = ["solo", "small_group", "large_group"]

def bin_passenger_count(
    df: pd.DataFrame,
    bins: list[int] = PASSENGER_BINS,
    labels: list[str] = PASSENGER_LABELS
) -> pd.DataFrame:
    """
    Bin ``passenger_count`` into categorical groups.

    Bin definitions:
      - solo : 1 passenger
      - small_group : 2-3 passengers
      - large_group : 4-9 passengers

    The raw ``passenger_count`` column is retained at this stage and
    dropped later in this phase, along with other raw columns.

    Parameters
    ----------
    df: pd.DataFrame
        DataFrame containing ``passenger_count`` as an integer column.
    bins: list[int]
        Bin edge values passed to ``pd.cut``. Must have len(labels) + 1
        elements.
    labels: list[str]
        Category labels for each bin.

    Returns
    -------
    pd.DataFrame
        Copy of the DataFrame with ``passenger_group`` (object dtype)
        appended.
    """
    df = df.copy()

    df["passenger_group"] = pd.cut(
        df["passenger_count"],
        bins = bins,
        labels = labels,
        right = True
    ).astype(str)

    counts = df["passenger_group"].value_counts()
    for label, count in counts.items():
        pct = count / len(df) * 100
        logger.info(f"  {label:<14} : {count:>8,}  ({pct:.1f}%)")

    return df

# Run
df = bin_passenger_count(df)

# Verify
print("\n"+ "=" * 75)
print("Passenger Group Distribution")
print("=" * 75)

vc = df["passenger_group"].value_counts()
for group, count in vc.items():
    pct = count / len(df) * 100
    print(f"  {group:<15} : {count:>8,} ({pct:.1f}%)")

print("\n"+ "=" * 75)
print("Average Fare by Passenger Group")
print("=" * 75)

fare_by_group = (
    df.groupby("passenger_group")["fare_amount"]
    .agg(["mean", "median", "count"])
    .round(2)
    .rename(columns={"mean": "avg_fare", "median": "med_fare", "count": "trips"})
)
print(fare_by_group.to_string())

print("\n"+ "=" * 75)
print("Unmapped values (should be 0)")
print("=" * 75)

unmapped = (df["passenger_group"] == "nan").sum()
print(f"  {unmapped:,} {'✅' if unmapped == 0 else '⚠️ investigate'}")


## Step 4.4: Passenger Count Binning ✅

`passenger_count` binned into three groups: solo (1), small_group (2–3), large_group (4–9). No unmapped values.

### Group Distribution

| Group | Trips | \% |
|---|---|---|
| `solo` | 696,021 | 70.2\% |
| `small_group` | 194,551 | 19.6\% |
| `large_group` | 101,426 | 10.2\% |

### Average Fare by Group

| Group | Avg Fare | Median Fare |
|---|---|---|
| `small_group` | \\$13.70 | \\$9.50 |
| `large_group` | \\$13.42 | \\$9.50 |
| `solo` | $12.88 | \\$9.00 |

**Observations:**
- Group rides command a modest fare premium (\\$13.42–\\$13.70) over solo rides (\\$12.88), consistent with the Phase 2 finding, but the
  difference is small enough (~\\$0.82) that this feature will contribute limited predictive power
- The median is identical across all three groups (\\$9.50), the mean difference is driven by a small number of high-fare group trips rather than a systematic group-size fare effect
- The binning reduces 9 raw values to 3 meaningful categories, removing noise from the near-zero Pearson r (0.016)

## Step 4.5: Categorical Encoding
One-hot encode `ratecodeid` and `vendorid`. Encode `payment_type` as a binary flag (credit card vs other). One-hot encode `passenger_group` from the previous step. Drop the first dummy column from each one-hot encoding to avoid multicollinearity (dummy variable trap).

In [ ]:
# ------------------------------------------------------------
# Step 4.5: Categorical Encoding
# ------------------------------------------------------------

# Rate code labels for readable column names
RATECODEID_MAP: dict[int, str] = {
    1: "standard",
    2: "jfk",
    3: "newark",
    4: "nassau_wc",
    5: "negotiated",
    6: "group_ride",
}

VENDORID_MAP: dict[int, str] = {
    1: "creative_mobile",
    2: "verifone",
}

def encode_ratecodeid(df: pd.DataFrame) -> pd.DataFrame:
    """
    One-hot encode ``ratecodeid`` into binary flag column.

    Each rate code becomes a column named ``rate_{label}``.
    ``rate_standard`` (code 1) is dropped as the reference category
    to avoid the dummy variable trap, its values is implied when all
    other rate flags are 0.

    Parameters
    ----------
    df: pd.DataFrame
        DataFrame containing ``ratecodeid`` as an integer column.

    Returns
    -------
    pd.DataFrame
        Copy of the DataFrame with rate code dummy columns appended.
        The raw ``ratecodeid`` column is retained for later steps.
    """
    df = df.copy()
    df["rate_label"] = df["ratecodeid"].map(RATECODEID_MAP)

    dummies = pd.get_dummies(
        df["rate_label"], prefix="rate", drop_first=False, dtype=int
    )

    # Drop reference category (standard rate, most common at 96.7%)
    if "rate_standard" in dummies.columns:
        dummies = dummies.drop(columns=["rate_standard"])

    df = pd.concat([df, dummies], axis=1)
    df = df.drop(columns=["rate_label"])

    added = [c for c in dummies.columns]
    logger.info(f"ratecodeid encoded -> {added}")

    return df

def encode_vendorid(df: pd.DataFrame) -> pd.DataFrame:
    """
    One-hot encode ``vendorid`` into a single binary flag.

    ``vendor_verifone`` = 1 for VeriFone Inc., 0 for Creative Mobile.
    ``vendor_creative_mobile`` is dropped as the reference category.

    Parameters
    ----------
    df: pd.DataFrame
        DataFrame containing ``vendorid`` as an integer column.

    Returns
    -------
    pd.DataFrame
        Copy of the DataFrame with ``vendor_verifone`` appended.
    """
    df = df.copy()
    df["vendor_label"] = df["vendorid"].map(VENDORID_MAP)

    dummies = pd.get_dummies(
        df["vendor_label"], prefix="vendor", drop_first=False, dtype=int
    )

    # Drop reference category
    if "vendor_creative_mobile" in dummies.columns:
        dummies = dummies.drop(columns=["vendor_creative_mobile"])

    df = pd.concat([df, dummies], axis=1)
    df = df.drop(columns=["vendor_label"])

    logger.info(f"vendorid encoded -> {dummies.columns.tolist()}")

    return df

def encode_passenger_group(df: pd.DataFrame) -> pd.DataFrame:
    """
    One-hot encode ``passenger_group`` into binary flag columns.

    ``pax_solo`` is dropped as the reference category (70.2% of trips).

    Parameters
    ----------
    df: pd.DataFrame
        DataFrame containing ``passenger_group`` as a string column.
        
    Returns
    -------
    pd.DataFrame
        Copy of the DataFrame with passenger group dummy columns appended.
    """
    df = df.copy()

    dummies = pd.get_dummies(
        df["passenger_group"], prefix="pax", drop_first=False, dtype=int
    )

    if "pax_solo" in dummies.columns:
        dummies = dummies.drop(columns=["pax_solo"])

    df = pd.concat([df, dummies], axis=1)

    logger.info(f"passenger_group encoded -> {dummies.columns.tolist()}")

    return df

def encode_payment_type(df: pd.DataFrame) -> pd.DataFrame:
    """
    Encode ``payment_type`` as a binary flag.

    ``is_credit_card`` = 1 for credit card payments (type 1), 0 for all
    other payment types (cash, no charge, dispute). This captures the only
    meaningful binary split in the payment type distribution.

    Parameters
    ----------
    df: pd.DataFrame
        DataFrame containing ``payment_type`` as an integer column.
    
    Returns
    -------
    pd.DataFrame
        Copy of the DataFrame with ``is_credit_card`` appended.
    """
    df = df.copy()
    df["is_credit_card"] = (df["payment_type"] == 1).astype(int)

    rate = df["is_credit_card"].mean() * 100
    logger.info(f"payment_type encoded -> 'is_credit_card' ({rate:.1f}% = 1)")

    return df

# Run all encoders
df = encode_ratecodeid(df)
df = encode_vendorid(df)
df = encode_passenger_group(df)
df = encode_payment_type(df)

# Verify
encoded_cols = [
    c for c in df.columns
    if c.startswith(("rate_", "vendor_", "pax_", "is_credit_card"))
]

print("\n" + "=" * 75)
print("Encoded Feature Summary")
print("=" * 75)

print(f"\n  {'Column':<25} {'dtype':<10} {'sum':>8} {'mean':>8}")
print("-" * 60)

for col in encoded_cols:
    print(
        f"  {col:<25} {str(df[col].dtype):<10} "
        f"{int(df[col].sum()):>8,} {df[col].mean():>8.4f}"
    )

print(f"\n  Total columns now: {df.shape[1]}")


## Step 4.5: Categorical Encoding ✅

Four encoding operations completed. Total columns now: **31**.

### Encoded Feature Summary

| Column | Dtype | Sum | Mean | Notes |
|---|---|---|---|---|
| `rate_group_ride` | `int64` | 2 | 0.0000 | Group ride code, negligible |
| `rate_jfk` | `int64` | 24,598 | 0.0248 | 2.5\% of trips |
| `rate_nassau_wc` | `int64` | 657 | 0.0007 | Negligible |
| `rate_negotiated` | `int64` | 1,777 | 0.0018 | Negligible |
| `rate_newark` | `int64` | 2,330 | 0.0023 | Negligible |
| `vendor_verifone` | `int64` | 557,950 | 0.5625 | 56.3\% of trips |
| `pax_large_group` | `int64` | 101,426 | 0.1022 | 10.2\% of trips |
| `pax_small_group` | `int64` | 194,551 | 0.1961 | 19.6\% of trips |
| `is_credit_card` | `int64` | 635,433 | 0.6406 | 64.1\% of trips |

### Reference Categories (dropped to avoid dummy variable trap)

| Original Column | Reference Category | Rationale |
|---|---|---|
| `ratecodeid` | `rate_standard` (code 1) | 96.7\% of trips — implied when all `rate_*` flags = 0 |
| `vendorid` | `vendor_creative_mobile` | Implied when `vendor_verifone` = 0 |
| `passenger_group` | `pax_solo` | 70.2\% of trips — implied when both `pax_*` flags = 0 |

**Observations:**
- All encoded sums and means match the Phase 2/Step 2.3 frequency tables: `vendor_verifone` (56.25\%) matches the 56.19% vendor split,
  `is_credit_card` (64.06%) matches the 63.86\% payment split (small differences due to row drops in Phase 3)
- `rate_group_ride` is now active for only **2 trips** (down from 10 in the raw data): most were removed during Phase 3 invalid-row cleaning. This feature carries virtually zero signal but is retained for completeness, tree-based models will naturally ignore it
- The five `rate_*` flags together let the model learn distinct intercepts for each flat-rate regime, directly addressing the JFK
  \\$52 flat-rate step-change identified in Phase 2
- Column count grew from 17 to 31, the increase reflects 5 rate flags + 1 vendor flag + 2 passenger group flags +
  1 payment flag + the 5 temporal/duration/log1p features added in earlier steps

## Step 4.6: Drop Remaining Unused Columns
Remove raw columns that have been superseded by engineered features, plus the location ID columns. With ~250+ unique zones in `pulocationid` and `dolocationid`, one-hot encoding would explode dimensionality, they are dropped rather than encoded, consistent with the Phase 4 plan.

In [ ]:
# ------------------------------------------------------------
# Step 4.6: Drop Remaining Unused Columns
# ------------------------------------------------------------

# Columns superseded by engineered features or dropped per Phase 4 plan
COLS_TO_DROP: list[str] = [
    # Raw datetimes superseded by temporal features
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    # Raw columns superseded by log1p versions
    "trip_distance",
    "tolls_amount",
    "trip_duration_min",
    # Raw columns superseded by encoded versions
    "passenger_count",
    "passenger_group",
    "ratecodeid",
    "vendorid",
    "payment_type",
    # High-cardinality location IDs, not encoded (250+ zones each)
    "pulocationid",
    "dolocationid",
]

def drop_engineered_columns(
    df: pd.DataFrame,
    cols: list[str] = COLS_TO_DROP
) -> pd.DataFrame:
    """
    Drop raw columns superseded by engineered features.

    Parameters
    ----------
    df: pd.DataFrame
        DataFrame with all Phase 4 features already created.
    cols: list[str]
        Column names to drop. Columns not present are skipped with
        a warning.

    Returns
    -------
    pd.DataFrame
        DataFrame with superseded columns removed.
    """
    cols_present = [c for c in cols if c in df.columns]
    cols_missing = [c for c in cols if c not in df.columns]

    if cols_missing:
        logger.warning(f"Columns not found (already dropped?): {cols_missing}")

    df_out = df.drop(columns=cols_present)

    logger.info(f"Dropped {len(cols_present)} columns: {cols_present}")
    logger.info(f"Columns remaining: {df_out.shape[1]}")

    return df_out

# Run
df = drop_engineered_columns(df)

# Verify
print("\n" + "=" * 75)
print("Final Feature Set")
print("=" * 75)

print(f"\n  Rows: {df.shape[0]:,}")
print(f"  Columns: {df.shape[1]}")

print("\n" + "=" * 75)
print("Column List")
print("=" * 75)

for i, col in enumerate(df.columns.tolist(), 1):
    print(f"  {i:>2}. {col:<25} {str(df[col].dtype)}")


## Step 4.6: Drop Remaining Unused Columns ✅

12 superseded raw columns dropped. Dataset reduced from 31 -> **19 columns**, 991,998 rows retained (no row drops in this step).

### Dropped Columns

| Column | Reason |
|---|---|
| `tpep_pickup_datetime` | Superseded by `pickup_hour`, `pickup_day_of_week`, `is_weekend`, `is_rush_hour`, `is_overnight` |
| `tpep_dropoff_datetime` | Used only to compute `trip_duration_min` |
| `trip_distance` | Superseded by `log1p_trip_distance` |
| `tolls_amount` | Superseded by `log1p_tolls_amount` |
| `trip_duration_min` | Superseded by `log1p_trip_duration_min` |
| `passenger_count` | Superseded by `passenger_group` |
| `passenger_group` | Superseded by `pax_small_group`, `pax_large_group` |
| `ratecodeid` | Superseded by `rate_*` flags |
| `vendorid` | Superseded by `vendor_verifone` |
| `payment_type` | Superseded by `is_credit_card` |
| `pulocationid` | High cardinality (250+ zones) — not encoded |
| `dolocationid` | High cardinality (250+ zones) — not encoded |

### Final Feature Set (19 columns)

| # | Column | Dtype | Category |
|---|---|---|---|
| 1 | `fare_amount` | `float64` | **Target** |
| 2 | `extra` | `float64` | Raw numeric (retained) |
| 3 | `pickup_hour` | `int32` | Temporal |
| 4 | `pickup_day_of_week` | `int32` | Temporal |
| 5 | `is_weekend` | `int64` | Temporal flag |
| 6 | `is_rush_hour` | `int64` | Temporal flag |
| 7 | `is_overnight` | `int64` | Temporal flag |
| 8 | `log1p_trip_distance` | `float64` | Log-transformed |
| 9 | `log1p_tolls_amount` | `float64` | Log-transformed |
| 10 | `log1p_trip_duration_min` | `float64` | Log-transformed |
| 11 | `rate_group_ride` | `int64` | One-hot (ratecodeid) |
| 12 | `rate_jfk` | `int64` | One-hot (ratecodeid) |
| 13 | `rate_nassau_wc` | `int64` | One-hot (ratecodeid) |
| 14 | `rate_negotiated` | `int64` | One-hot (ratecodeid) |
| 15 | `rate_newark` | `int64` | One-hot (ratecodeid) |
| 16 | `vendor_verifone` | `int64` | One-hot (vendorid) |
| 17 | `pax_large_group` | `int64` | One-hot (passenger_group) |
| 18 | `pax_small_group` | `int64` | One-hot (passenger_group) |
| 19 | `is_credit_card` | `int64` | Binary (payment_type) |

> 18 features + 1 target. The feature set spans temporal, distance/duration,
> rate-regime, vendor, group-size, and payment dimensions, covering every
> signal identified during Phase 2 EDA.

## Step 4.7: Post-Engineering Validation
Run a final validation pass on the engineered dataset: confirm no missing values, verify dtypes are model-ready (no object columns), check value ranges for binary/dummy columns, and re-run the correlation analysis against `fare_amount` to see how the engineered features compare to the raw features from Phase 2.

In [ ]:
# ------------------------------------------------------------
# Step 4.7: Post-Engineering Validation
# ------------------------------------------------------------

def validate_engineered_dataset(df: pd.DataFrame, target_col: str = TARGET_COL) -> pd.Series:
    """
    Run a full validation suite on the engineered feature matrix.

    Checks performed:
      1. Missing values
      2. Dtype check — confirms no ``object`` columns remain
      3. Binary/dummy column range check — confirms values are in {0, 1}
      4. Feature correlation with the target, sorted by |r|

    Parameters
    ----------
    df: pd.DataFrame
        The engineered feature matrix.
    target_col: str
        Name of the target column (default ``TARGET_COL``).

    Returns
    -------
    pd.Series
        Feature correlations with the target, sorted by absolute value
        descending.
    """
    print("\n" + "=" * 75)
    print("1. Missing Values")
    print("=" * 75)
    
    total_nulls = df.isna().sum().sum()
    if total_nulls == 0:
        print("  ✅ No missing values detected.")
    else:
        print(f"  ⚠️  {total_nulls:,} missing values remain:")
        print(df.isna().sum()[df.isna().sum() > 0].to_string())

    print("\n" + "=" * 75)
    print("2. Dtype Check")
    print("=" * 75)
    
    object_cols = df.select_dtypes(include="object").columns.tolist()
    if not object_cols:
        print("  ✅ No object dtype columns — fully numeric matrix.")
    else:
        print(f"  ⚠️  Object columns remain: {object_cols}")

    print("\n" + "=" * 75)
    print("3. Binary/Dummy Column Range Check")
    print("=" * 75)
    
    binary_prefixes = ("rate_", "vendor_", "pax_", "is_")
    binary_cols = [c for c in df.columns if c.startswith(binary_prefixes)]
    all_binary_ok = True
    for col in binary_cols:
        unique_vals = set(df[col].unique())
        ok = unique_vals.issubset({0, 1})
        if not ok:
            all_binary_ok = False
            print(f"  ⚠️  {col}: unexpected values {unique_vals}")
    if all_binary_ok:
        print(f"  ✅ All {len(binary_cols)} binary/dummy columns contain only {{0, 1}}.")

    print("\n" + "=" * 75)
    print("4. Feature Correlation with fare_amount")
    print("=" * 75)
    
    corr = df.corr(numeric_only=True)[target_col].drop(labels=[target_col])
    corr_sorted = corr.reindex(corr.abs().sort_values(ascending=False).index)

    print(f"\n  {'Feature':<25} {'Pearson r':>10}  {'Strength'}")
    print(f"  {'─' * 50}")
    for feat, val in corr_sorted.items():
        strength = (
            "Strong"   if abs(val) >= 0.5  else
            "Moderate" if abs(val) >= 0.3  else
            "Weak"     if abs(val) >= 0.1  else
            "Negligible"
        )
        direction = "positive" if val > 0 else "negative"
        print(f"  {feat:<25} {val:>10.4f}  {strength} {direction}")

    return corr_sorted

# Run
target_corr_final = validate_engineered_dataset(df)

print("\n" + "=" * 75)
print("Final Shape")
print("=" * 75)
print(f"  Rows: {df.shape[0]:,}")
print(f"  Columns: {df.shape[1]}")
print(f"  Features: {df.shape[1] - 1}  (excluding target)")


## Step 4.7: Post-Engineering Validation ✅

All four validation checks passed. The feature matrix is fully numeric, complete, and model-ready.

### Validation Results

| Check | Result |
|---|---|
| Missing values | ✅ None |
| Object dtype columns | ✅ None — fully numeric matrix |
| Binary/dummy range {0,1} | ✅ All 12 binary columns clean |
| Feature count | 18 features + 1 target |

### Feature Correlations with `fare_amount`

| Feature | Pearson r | Strength | Notes |
|---|---|---|---|
| `log1p_trip_distance` | 0.8873 | Strong positive | Primary fare driver |
| `log1p_trip_duration_min` | 0.7428 | Strong positive | Strong secondary signal |
| `log1p_tolls_amount` | 0.6207 | Strong positive | Proxy for long trips |
| `rate_jfk` | 0.5417 | Strong positive | $52 flat rate drives correlation |
| `rate_newark` | 0.2361 | Weak positive | Outer-borough flat rate |
| `rate_negotiated` | 0.2283 | Weak positive | High-variance negotiated fares |
| `rate_nassau_wc` | 0.1219 | Weak positive | Outer-borough flat rate |
| `extra` | 0.0544 | Negligible | Rush hour/overnight surcharge |
| `is_credit_card` | 0.0466 | Negligible | No direct fare effect |
| `is_rush_hour` | −0.0276 | Negligible | Negative — short commute trips |
| `pax_small_group` | 0.0261 | Negligible | Marginal group premium |
| `vendor_verifone` | 0.0172 | Negligible | No vendor fare difference |
| `pickup_day_of_week` | 0.0106 | Negligible | Weak daily pattern |
| `pax_large_group` | 0.0096 | Negligible | Marginal group premium |
| `is_weekend` | 0.0046 | Negligible | Captured by day_of_week |
| `pickup_hour` | 0.0028 | Negligible | Captured by rush/overnight flags |
| `is_overnight` | 0.0017 | Negligible | Low linear signal |
| `rate_group_ride` | −0.0013 | Negligible | Only 2 trips, no signal |

**Observations:**
- The three log-transformed continuous features dominate — `log1p_trip_distance` (0.887), `log1p_trip_duration_min` (0.743), and `log1p_tolls_amount` (0.621) are the only features with strong linear relationships to fare
- `rate_jfk` (0.542) is the strongest categorical predictor by a wide margin, confirming the $52 flat-rate step-change identified in Phase 2
- The temporal flags (`is_rush_hour`, `is_overnight`, `pickup_hour`, `is_weekend`) all show negligible Pearson r — their value lies in non-linear interactions that tree-based models will capture but linear
  correlation cannot measure
- `rate_group_ride` (2 trips, r=−0.0013) is a confirmed dead feature, Random Forest and XGBoost will assign it zero importance; it will not affect model quality
- No feature shows a strong negative correlation, which means there are no features that actively suppress fare prediction

## Step 4.8: Save Engineered Dataset
Persist the final feature matrix to `data/taxi_features.parquet`. This is the direct input to Phase 5, no further transformation is needed before the train/test split.

In [ ]:
# ------------------------------------------------------------
# Step 4.8: Save Engineered Dataset
# ------------------------------------------------------------

FEATURES_PATH: Path = DATA_DIR / "taxi_features.parquet"


def save_features_parquet(
    df: pd.DataFrame,
    path: Path = FEATURES_PATH,
) -> None:
    """
    Save the engineered feature matrix to a Parquet file.

    Parameters
    ----------
    df: pd.DataFrame
        The fully engineered DataFrame to persist.
    path: Path
        Destination path (default ``DATA_DIR / "taxi_features.parquet"``).
    """
    df.to_parquet(path, index=False, compression="snappy")

    size_mb = path.stat().st_size / 1e6
    logger.info(f"Features saved → {path}")
    logger.info(f"File size on disk : {size_mb:.1f} MB")


def verify_features_roundtrip(path: Path = FEATURES_PATH) -> pd.DataFrame:
    """
    Reload and verify the saved features Parquet file.

    Confirms row count, column count, dtype consistency, and that the
    target column is present before Phase 5 begins.

    Parameters
    ----------
    path: Path
        Path to the saved Parquet file.

    Returns
    -------
    pd.DataFrame
        The reloaded DataFrame.
    """
    df_loaded = pd.read_parquet(path)
    logger.info(f"Features reloaded : {df_loaded.shape[0]:,} rows "
                f"× {df_loaded.shape[1]} columns")
    return df_loaded

# Save
save_features_parquet(df)

# Reload & verify
df_reloaded = verify_features_roundtrip()

print("\n" + "=" * 75)
print("Parquet Roundtrip Verification")
print("=" * 75)

print(f"\n  In-memory shape: {df.shape[0]:,} rows × {df.shape[1]} cols")
print(f"  Reloaded shape: {df_reloaded.shape[0]:,} rows × {df_reloaded.shape[1]} cols")

shape_match = df.shape == df_reloaded.shape
dtype_match = (df.dtypes == df_reloaded.dtypes).all()
target_present = TARGET_COL in df_reloaded.columns

print(f"\n  Shape match: {'✅' if shape_match else '⚠️  MISMATCH'}")
print(f"  Dtype match: {'✅' if dtype_match else '⚠️  MISMATCH'}")
print(f"  Target present: {'✅' if target_present else '⚠️  MISSING'}")

print("\n" + "=" * 75)
print(f"Reloaded Dtypes")
print("=" * 75)

print(df_reloaded.dtypes.to_string())
    

## Step 4.8: Save Engineered Dataset ✅

Feature matrix saved to `data/taxi_features.parquet` and verified with a full roundtrip reload.

### Parquet Roundtrip Verification

| Check | Result |
|---|---|
| Shape match | ✅ 991,998 rows × 19 columns |
| Dtype match | ✅ All 19 columns identical |
| Target present | ✅ `fare_amount` confirmed |

### Final Dtypes

| Column | Dtype |
|---|---|
| `fare_amount` | `float64` |
| `extra` | `float64` |
| `pickup_hour` | `int32` |
| `pickup_day_of_week` | `int32` |
| `is_weekend` | `int64` |
| `is_rush_hour` | `int64` |
| `is_overnight` | `int64` |
| `log1p_trip_distance` | `float64` |
| `log1p_tolls_amount` | `float64` |
| `log1p_trip_duration_min` | `float64` |
| `rate_group_ride` | `int64` |
| `rate_jfk` | `int64` |
| `rate_nassau_wc` | `int64` |
| `rate_negotiated` | `int64` |
| `rate_newark` | `int64` |
| `vendor_verifone` | `int64` |
| `pax_large_group` | `int64` |
| `pax_small_group` | `int64` |
| `is_credit_card` | `int64` |